In [1]:
import zipfile

import pandas as pd

from isochrone.download import RAW_DATA_DIR

FEED_PATH = RAW_DATA_DIR / "mvv_gtfs_2026-11-01.zip"
feed = zipfile.ZipFile(FEED_PATH)
feed.namelist()


['agency.txt',
 'calendar.txt',
 'calendar_dates.txt',
 'feed_info.txt',
 'routes.txt',
 'shapes.txt',
 'stops.txt',
 'stop_times.txt',
 'ticketing_deep_links.txt',
 'ticketing_identifiers.txt',
 'trips.txt']

In [2]:
from isochrone.gtfs import read_gtfs_table

read_gtfs_table(FEED_PATH, "feed_info")


,feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version
0,Münchner VerkehrsVerbund (MVV),https://www.mvv-auskunft.de,DE,20261001,20261213,20261005


In [3]:
import datetime as dt

from isochrone.gtfs import active_service_ids

trips = read_gtfs_table(FEED_PATH, "trips", usecols=["trip_id", "service_id"])
trips_per_service = trips["service_id"].value_counts()

dates = pd.date_range("2026-11-02", "2026-12-13").date
counts = {
    date: trips_per_service.reindex(list(active_service_ids(FEED_PATH, date)), fill_value=0).sum()
    for date in dates
}
daily = pd.DataFrame({"trips": counts})
daily["weekday"] = [date.strftime("%a") for date in daily.index]
daily


,trips,weekday
2026-11-02,40830,Mon
2026-11-03,40830,Tue
2026-11-04,40814,Wed
2026-11-05,40807,Thu
2026-11-06,40980,Fri
2026-11-07,30588,Sat
2026-11-08,21381,Sun
2026-11-09,43605,Mon
2026-11-10,43543,Tue
2026-11-11,43522,Wed


In [4]:
from isochrone.gtfs import gtfs_time_to_seconds

stop_times = read_gtfs_table(
    FEED_PATH,
    "stop_times",
    usecols=["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
)
arr = gtfs_time_to_seconds(stop_times["arrival_time"])
dep = gtfs_time_to_seconds(stop_times["departure_time"])

pd.Series(
    {
        "rows": len(stop_times),
        "min departure (h)": round(dep.min() / 3600, 2),
        "max departure (h)": round(dep.max() / 3600, 2),
        "departures >= 24 h": (dep >= 24 * 3600).sum(),
        "dwell > 0": (dep > arr).sum(),
        "departure < arrival": (dep < arr).sum(),
    },
    dtype=object,
)


rows                   2217866
min departure (h)          0.0
max departure (h)        30.13
departures >= 24 h       77946
dwell > 0               288091
departure < arrival          0
dtype: object

In [5]:
from isochrone.connections import build_connections

TEST_DATE = dt.date(2026, 12, 9)

connections = build_connections(FEED_PATH, TEST_DATE)
duration_min = (connections["arr_time"] - connections["dep_time"]) / 60

print(len(connections), "connections from", connections["trip_id"].nunique(), "trips")
print((duration_min == 0).sum(), "connections with zero duration")
no_board = (~connections["can_board"]).sum()
no_alight = (~connections["can_alight"]).sum()
print(no_board, "no boarding,", no_alight, "no alighting")
duration_min.describe().round(1)


769125 connections from 41788 trips
7060 connections with zero duration
3340 no boarding, 388 no alighting


count    769125.0
mean          1.7
std           1.4
min           0.0
25%           1.0
50%           1.2
75%           2.0
max          53.0
dtype: float64

In [6]:
from isochrone.transfers import load_platforms

platforms = load_platforms(FEED_PATH)
print(len(platforms))
platforms[["x", "y"]].describe().round(0)


18861


,x,y
count,18861.0,18861.0
mean,701121.0,5336735.0
std,32491.0,31886.0
min,551176.0,5251164.0
25%,680841.0,5316596.0
50%,697663.0,5335605.0
75%,728333.0,5354535.0
max,802884.0,5576986.0
